# C-Index STEP=2 거래비용 평가 전용 노트북

이 노트북은 모델 학습, XAI 추출, C-index 재계산을 하지 않는다.  
이미 생성된 pkl 캐시 파일을 불러와서 거래비용 반영 성과만 계산한다.

필요한 캐시 파일:

```text
/content/drive/MyDrive/c-index/artifacts/finalDf_cache.pkl
/content/drive/MyDrive/c-index/artifacts/oep_signals_cache.pkl
/content/drive/MyDrive/c-index/artifacts/analysisDf_cache.pkl
```



In [1]:
# Colab Drive mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TRADING_DAYS_PER_YEAR = 252
SHARPE_EPSILON = 1e-6
MIN_TRADES_POOLED = 15

CACHE_DIR = Path('/content/drive/MyDrive/c-index/artifacts')
FINAL_DF_CACHE = CACHE_DIR / 'finalDf_cache.pkl'
OEP_SIGNALS_CACHE = CACHE_DIR / 'oep_signals_cache.pkl'
ANALYSIS_DF_CACHE = CACHE_DIR / 'analysisDf_cache.pkl'

OUTPUT_DIR = Path('/content/drive/MyDrive/c-index/output_images')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not FINAL_DF_CACHE.exists():
    raise FileNotFoundError(f'finalDf cache not found: {FINAL_DF_CACHE}')

finalDf = pd.read_pickle(FINAL_DF_CACHE)
finalDf['date'] = pd.to_datetime(finalDf['date'])
finalDf = finalDf.sort_values(['model', 'asset', 'date']).reset_index(drop=True)

# Optional caches, useful for inspection only.
oep_signals = pd.read_pickle(OEP_SIGNALS_CACHE) if OEP_SIGNALS_CACHE.exists() else None
analysisDf = pd.read_pickle(ANALYSIS_DF_CACHE) if ANALYSIS_DF_CACHE.exists() else None

print(f'[cache loaded] finalDf: {FINAL_DF_CACHE} | shape={finalDf.shape}')
print('models:', sorted(finalDf['model'].unique()))
print('assets:', sorted(finalDf['asset'].unique()))
print('date range:', finalDf['date'].min(), 'to', finalDf['date'].max())




[cache loaded] finalDf: /content/drive/MyDrive/c-index/artifacts/finalDf_cache.pkl | shape=(861, 12)
models: ['gbm', 'mlp', 'rf']
assets: ['GLD', 'SPY', 'TLT']
date range: 2024-06-05 00:00:00 to 2025-12-24 00:00:00


In [3]:
def _prepare_trade_frame(trades, return_col='ret_1d_next', date_col='date'):
    if isinstance(trades, pd.DataFrame):
        out = trades.copy()
        if return_col not in out.columns:
            raise KeyError(f"'{return_col}' column is required for metric calculation.")
        out = out.rename(columns={return_col: 'ret'})
        out['date'] = pd.to_datetime(out[date_col]) if date_col in out.columns else pd.NaT
        return out[['date', 'ret']].dropna(subset=['ret'])

    out = pd.DataFrame({'ret': pd.Series(trades).dropna()})
    out['date'] = pd.NaT
    return out[['date', 'ret']]


def _max_drawdown(return_series):
    returns = pd.Series(return_series).dropna()
    if returns.empty:
        return np.nan
    equity = returns.cumsum()
    return float((equity - equity.cummax()).min() * 100)


def compute_metrics(trades, min_trades=MIN_TRADES_POOLED, calendar_dates=None):
    trade_df = _prepare_trade_frame(trades)
    if trade_df['date'].notna().any():
        trade_df = trade_df.sort_values('date').reset_index(drop=True)

    rets = trade_df['ret'].dropna()
    n = len(rets)

    if calendar_dates is not None:
        eval_days = len(pd.to_datetime(pd.Series(calendar_dates).dropna().unique()))
    elif trade_df['date'].notna().any():
        eval_days = len(pd.to_datetime(trade_df['date'].dropna().unique()))
    else:
        eval_days = np.nan

    annual_trades = n / (eval_days / TRADING_DAYS_PER_YEAR) if pd.notna(eval_days) and eval_days > 0 else np.nan

    if n == 0:
        return {
            'Trade Count': 0,
            'Avg Return': np.nan,
            'Win Rate': np.nan,
            'Sharpe': np.nan,
            'Sharpe_annualized': np.nan,
            'Annual Trades': float(annual_trades) if pd.notna(annual_trades) else np.nan,
            'Eval Days': int(eval_days) if pd.notna(eval_days) else np.nan,
            'Max Drawdown': np.nan,
            'Note': ''
        }

    avg_ret = rets.mean() * 100
    win_rate = (rets > 0).mean() * 100
    mdd = _max_drawdown(rets)

    if n < min_trades:
        sharpe = np.nan
        sharpe_ann = np.nan
        note = f'N={n}<{min_trades} (소표본 경고: Sharpe 신뢰 불가)'
    else:
        std = rets.std()
        if std <= SHARPE_EPSILON:
            sharpe = np.nan
            sharpe_ann = np.nan
            note = '표준편차≈0 (Sharpe 신뢰 불가)'
        else:
            sharpe = float(rets.mean() / std)
            sharpe_ann = float(sharpe * np.sqrt(annual_trades)) if pd.notna(annual_trades) else np.nan
            note = ''

    return {
        'Trade Count': int(n),
        'Avg Return': float(avg_ret),
        'Win Rate': float(win_rate),
        'Sharpe': sharpe,
        'Sharpe_annualized': sharpe_ann,
        'Annual Trades': float(annual_trades) if pd.notna(annual_trades) else np.nan,
        'Eval Days': int(eval_days) if pd.notna(eval_days) else np.nan,
        'Max Drawdown': float(mdd),
        'Note': note
    }


def apply_transaction_cost(trades, cost_bps=0):
    out = trades.copy()
    out['ret_1d_next'] = out['ret_1d_next'] - (cost_bps / 10000.0)
    return out



In [4]:
q_list = [0.2, 0.3, 0.4, 0.5]
cost_bps_list = [0, 5, 10]
c_index_cols = finalDf.filter(like='C_', axis=1)

if c_index_cols.empty:
    raise ValueError('No C-index columns found in finalDf.')

print('C-index columns:', list(c_index_cols.columns))



C-index columns: ['C_tau_full', 'C_tau_ties3', 'C_tau_ties5', 'C_tau_ties10', 'C_RBO_full', 'C_Mallows_full']


In [5]:
def evaluate_no_filter_with_cost(df, cost_bps=0, min_trades=MIN_TRADES_POOLED):
    df = df.copy().sort_values('date')
    trades = df.loc[df['y_hat'] == 1, ['date', 'ret_1d_next']].copy()
    net_trades = apply_transaction_cost(trades, cost_bps=cost_bps)
    metrics = compute_metrics(net_trades, min_trades=min_trades, calendar_dates=df['date'].unique())
    metrics['C-Index Type'] = 'No Filter'
    metrics['Quantile'] = np.nan
    metrics['Cost (bp)'] = cost_bps
    return metrics


def evaluate_filter_with_cost(df, c_col, quantile=0.3, cost_bps=0, min_trades=MIN_TRADES_POOLED):
    df = df.copy().sort_values('date')
    threshold = df[c_col].quantile(quantile)
    is_trade = (df['y_hat'] == 1) & (df[c_col] >= threshold)
    trades = df.loc[is_trade, ['date', 'ret_1d_next']].copy()
    net_trades = apply_transaction_cost(trades, cost_bps=cost_bps)
    metrics = compute_metrics(net_trades, min_trades=min_trades, calendar_dates=df['date'].unique())
    metrics['C-Index Type'] = c_col
    metrics['Quantile'] = quantile
    metrics['Threshold Value'] = threshold
    metrics['Cost (bp)'] = cost_bps
    return metrics


results = []

for cost_bps in cost_bps_list:
    for model_name, sub in finalDf.groupby('model'):
        m = evaluate_no_filter_with_cost(sub, cost_bps=cost_bps)
        m['Model'] = model_name
        results.append(m)

    for model_name, sub in finalDf.groupby('model'):
        for col in c_index_cols.columns:
            for q in q_list:
                m = evaluate_filter_with_cost(sub, col, quantile=q, cost_bps=cost_bps)
                m['Model'] = model_name
                results.append(m)

summaryTableCost = pd.DataFrame(results)

valid_pool_cost = summaryTableCost[
    (summaryTableCost['C-Index Type'] != 'No Filter') &
    (summaryTableCost['Trade Count'] >= MIN_TRADES_POOLED)
]

best_rows_cost = (
    valid_pool_cost
    .sort_values(['Cost (bp)', 'Model', 'Sharpe'], ascending=[True, True, False])
    .groupby(['Cost (bp)', 'Model'])
    .first()
    .reset_index()
)

display(summaryTableCost)
display(best_rows_cost)



,Trade Count,Avg Return,Win Rate,Sharpe,Sharpe_annualized,Annual Trades,Eval Days,Max Drawdown,Note,C-Index Type,Quantile,Cost (bp),Model,Threshold Value
0,67,0.313659,61.194030,0.185403,1.631644,77.449541,218,-6.134243,,No Filter,NaN,0,gbm,NaN
1,116,0.217741,59.482759,0.138153,1.599784,134.091743,218,-9.376242,,No Filter,NaN,0,mlp,NaN
2,172,0.200919,57.558140,0.132599,1.869715,198.825688,218,-13.807412,,No Filter,NaN,0,rf,NaN
3,67,0.313659,61.194030,0.185403,1.631644,77.449541,218,-6.134243,,C_tau_full,0.2,0,gbm,0.000000
4,67,0.313659,61.194030,0.185403,1.631644,77.449541,218,-6.134243,,C_tau_full,0.3,0,gbm,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,91,0.188764,60.439560,0.109405,1.122091,105.192661,218,-10.833708,,C_RBO_full,0.5,10,rf,0.403627
221,172,0.100919,57.558140,0.066603,0.939132,198.825688,218,-15.407412,,C_Mallows_full,0.2,10,rf,0.606531
222,172,0.100919,57.558140,0.066603,0.939132,198.825688,218,-15.407412,,C_Mallows_full,0.3,10,rf,0.606531
223,172,0.100919,57.558140,0.066603,0.939132,198.825688,218,-15.407412,,C_Mallows_full,0.4,10,rf,0.606531


,Cost (bp),Model,Trade Count,Avg Return,Win Rate,Sharpe,Sharpe_annualized,Annual Trades,Eval Days,Max Drawdown,Note,C-Index Type,Quantile,Threshold Value
0,0,gbm,32,0.373487,68.750000,0.321185,1.953451,36.990826,218,-2.955633,,C_RBO_full,0.5,0.463832
1,0,mlp,66,0.291674,66.666667,0.221164,1.931782,76.293578,218,-7.961615,,C_tau_ties10,0.4,0.084211
2,0,rf,97,0.325110,61.855670,0.190639,2.018690,112.128440,218,-10.993207,,C_tau_ties3,0.5,0.102564
3,5,gbm,32,0.323487,68.750000,0.278187,1.691936,36.990826,218,-3.005633,,C_RBO_full,0.5,0.463832
4,5,mlp,66,0.241674,66.666667,0.183251,1.600628,76.293578,218,-8.461615,,C_tau_ties10,0.4,0.084211
5,5,rf,97,0.275110,61.855670,0.161320,1.708228,112.128440,218,-11.293207,,C_tau_ties3,0.5,0.102564
6,10,gbm,32,0.273487,68.750000,0.235189,1.430420,36.990826,218,-3.055633,,C_RBO_full,0.5,0.463832
7,10,mlp,66,0.191674,66.666667,0.145338,1.269474,76.293578,218,-8.961615,,C_tau_ties10,0.4,0.084211
8,10,rf,97,0.225110,61.855670,0.132001,1.397765,112.128440,218,-11.593207,,C_tau_ties3,0.5,0.102564


In [6]:
baseline_cost = summaryTableCost[summaryTableCost['C-Index Type'] == 'No Filter'][
    ['Cost (bp)', 'Model', 'Trade Count', 'Avg Return', 'Win Rate', 'Sharpe',
     'Sharpe_annualized', 'Annual Trades', 'Max Drawdown']
].rename(columns={
    'Trade Count': 'Baseline Trade Count',
    'Avg Return': 'Baseline Avg Return',
    'Win Rate': 'Baseline Win Rate',
    'Sharpe': 'Baseline Sharpe',
    'Sharpe_annualized': 'Baseline Sharpe Annualized',
    'Annual Trades': 'Baseline Annual Trades',
    'Max Drawdown': 'Baseline Max Drawdown'
})

costImprovementTable = pd.merge(best_rows_cost, baseline_cost, on=['Cost (bp)', 'Model'], how='left')
costImprovementTable['Delta Trade Count'] = costImprovementTable['Trade Count'] - costImprovementTable['Baseline Trade Count']
costImprovementTable['Delta Avg Return'] = costImprovementTable['Avg Return'] - costImprovementTable['Baseline Avg Return']
costImprovementTable['Delta Win Rate'] = costImprovementTable['Win Rate'] - costImprovementTable['Baseline Win Rate']
costImprovementTable['Delta Sharpe'] = costImprovementTable['Sharpe'] - costImprovementTable['Baseline Sharpe']
costImprovementTable['Delta Sharpe Annualized'] = costImprovementTable['Sharpe_annualized'] - costImprovementTable['Baseline Sharpe Annualized']
costImprovementTable['Delta MDD'] = costImprovementTable['Max Drawdown'] - costImprovementTable['Baseline Max Drawdown']

costImprovementTable = costImprovementTable[[
    'Cost (bp)', 'Model', 'C-Index Type', 'Quantile',
    'Baseline Trade Count', 'Trade Count', 'Delta Trade Count',
    'Baseline Avg Return', 'Avg Return', 'Delta Avg Return',
    'Baseline Win Rate', 'Win Rate', 'Delta Win Rate',
    'Baseline Sharpe', 'Sharpe', 'Delta Sharpe',
    'Baseline Sharpe Annualized', 'Sharpe_annualized', 'Delta Sharpe Annualized',
    'Baseline Max Drawdown', 'Max Drawdown', 'Delta MDD',
    'Annual Trades', 'Eval Days', 'Note'
]].sort_values(['Cost (bp)', 'Model']).reset_index(drop=True)

display(costImprovementTable.round(4))



,Cost (bp),Model,C-Index Type,Quantile,Baseline Trade Count,Trade Count,Delta Trade Count,Baseline Avg Return,Avg Return,Delta Avg Return,...,Delta Sharpe,Baseline Sharpe Annualized,Sharpe_annualized,Delta Sharpe Annualized,Baseline Max Drawdown,Max Drawdown,Delta MDD,Annual Trades,Eval Days,Note
0,0,gbm,C_RBO_full,0.5,67,32,-35,0.3137,0.3735,0.0598,...,0.1358,1.6316,1.9535,0.3218,-6.1342,-2.9556,3.1786,36.9908,218,
1,0,mlp,C_tau_ties10,0.4,116,66,-50,0.2177,0.2917,0.0739,...,0.0830,1.5998,1.9318,0.3320,-9.3762,-7.9616,1.4146,76.2936,218,
2,0,rf,C_tau_ties3,0.5,172,97,-75,0.2009,0.3251,0.1242,...,0.0580,1.8697,2.0187,0.1490,-13.8074,-10.9932,2.8142,112.1284,218,
3,5,gbm,C_RBO_full,0.5,67,32,-35,0.2637,0.3235,0.0598,...,0.1223,1.3715,1.6919,0.3204,-6.4342,-3.0056,3.4286,36.9908,218,
4,5,mlp,C_tau_ties10,0.4,116,66,-50,0.1677,0.2417,0.0739,...,0.0768,1.2324,1.6006,0.3682,-9.9762,-8.4616,1.5146,76.2936,218,
5,5,rf,C_tau_ties3,0.5,172,97,-75,0.1509,0.2751,0.1242,...,0.0617,1.4044,1.7082,0.3038,-14.6074,-11.2932,3.3142,112.1284,218,
6,10,gbm,C_RBO_full,0.5,67,32,-35,0.2137,0.2735,0.0598,...,0.1089,1.1114,1.4304,0.3190,-6.8070,-3.0556,3.7514,36.9908,218,
7,10,mlp,C_tau_ties10,0.4,116,66,-50,0.1177,0.1917,0.0739,...,0.0706,0.8651,1.2695,0.4044,-10.5762,-8.9616,1.6146,76.2936,218,
8,10,rf,C_tau_ties3,0.5,172,97,-75,0.1009,0.2251,0.1242,...,0.0654,0.9391,1.3978,0.4586,-15.4074,-11.5932,3.8142,112.1284,218,


In [7]:
def save_df_pretty(df, filename, dpi=450, title=None, include_index=False):
    df_show = df.copy()
    if include_index:
        df_show = df_show.reset_index()
    nrows, ncols = df_show.shape
    fig_w = max(10, ncols * 1.8)
    fig_h = max(2.5, (nrows + 1) * 0.52)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=13, pad=14)
    tbl = ax.table(cellText=df_show.values, colLabels=df_show.columns, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.08, 1.55)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(1.1)
        if r == 0:
            cell.set_text_props(weight='bold')
            cell.set_height(cell.get_height() * 1.12)
    plt.tight_layout(pad=1.5)
    fig.savefig(OUTPUT_DIR / filename, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)

save_df_pretty(summaryTableCost, 'cost_summary_performance_table.png', title='Cost-aware Summary Performance Table')
save_df_pretty(best_rows_cost, 'cost_best_scenarios_by_model.png', title='Best Scenarios by Model and Cost')
save_df_pretty(costImprovementTable.round(4), 'cost_improvement_table.png', title='Cost-aware Improvement over No Filter')

print(f'saved cost-aware tables to {OUTPUT_DIR}')



saved cost-aware tables to /content/drive/MyDrive/c-index/output_images


## Trade-quality statistical test

이 섹션은 위에서 만든 거래비용 반영 결과표(`summaryTableCost`, `best_rows_cost`, `costImprovementTable`) 아래에 이어서 실행한다.

검정 질문은 row-level 평균 수익률이 아니라, 교수님 피드백의 핵심인 다음 문제에 맞춘다.

> 6개 C-index 변형과 4개 q 후보를 탐색한 뒤 best를 고른 효과를 보정해도, active C-index filter가 No Filter 대비 per-trade 성과를 개선하는가?

따라서 main selection metric은 `Delta Sharpe`로 두고, active filter 후보만 best 선택에 포함한다.

- `Trade Count >= 15`: 소표본 Sharpe 폭발 방지
- `Delta Trade Count < 0`: 실제로 거래를 줄인 active filter만 포함
- `Delta Sharpe`: 교수님이 권고한 per-trade Sharpe 중심 지표
- `Delta Avg Return`, `Delta MDD`, `Delta Win Rate`: 보조 trade-quality 지표

Block bootstrap은 STEP=2 rolling window overlap을 고려해 20 trading-day block 단위로 신뢰구간을 계산한다. Block permutation + max-stat correction은 24개 후보 중 best를 사후 선택한 multiple testing 문제를 보정한다.

In [8]:
# Trade-quality statistical test settings

SEED = 42
BLOCK_LENGTH_DAYS = 20
N_BOOTSTRAP = 2000
N_PERMUTATIONS = 2000

# Main metric for best-candidate selection and max-stat correction.
SELECTION_METRIC = 'Delta Sharpe'

# Active filter conditions.
ACTIVE_MIN_TRADES = MIN_TRADES_POOLED

np.random.seed(SEED)

print('SELECTION_METRIC:', SELECTION_METRIC)
print('BLOCK_LENGTH_DAYS:', BLOCK_LENGTH_DAYS)
print('N_BOOTSTRAP:', N_BOOTSTRAP)
print('N_PERMUTATIONS:', N_PERMUTATIONS)
print('ACTIVE_MIN_TRADES:', ACTIVE_MIN_TRADES)

SELECTION_METRIC: Delta Sharpe
BLOCK_LENGTH_DAYS: 20
N_BOOTSTRAP: 2000
N_PERMUTATIONS: 2000
ACTIVE_MIN_TRADES: 15


### Candidate grid 재평가

위 거래비용 표는 이미 계산되어 있지만, 통계검정에서는 후보별 baseline 대비 개선폭과 active/no-op 여부가 필요하다.

여기서는 model-cost pair마다 24개 후보(`6 C-index x 4 q`)를 다시 정리하고, 다음 조건을 만족하는 후보만 best 선택 대상으로 둔다.

```text
Trade Count >= 15
Delta Trade Count < 0
Sharpe is finite
```

즉 MLP의 일부 `C_tau_full`처럼 거래 수를 전혀 줄이지 않는 no-op 후보는 main best 후보에서 제외된다.

In [9]:
def _with_baseline_deltas(candidate_df, baseline_row):
    out = candidate_df.copy()
    base = baseline_row.copy()

    rename_map = {
        'Trade Count': 'Baseline Trade Count',
        'Avg Return': 'Baseline Avg Return',
        'Win Rate': 'Baseline Win Rate',
        'Sharpe': 'Baseline Sharpe',
        'Sharpe_annualized': 'Baseline Sharpe Annualized',
        'Max Drawdown': 'Baseline Max Drawdown'
    }

    for src, dst in rename_map.items():
        out[dst] = base[src]

    out['Delta Trade Count'] = out['Trade Count'] - out['Baseline Trade Count']
    out['Delta Avg Return'] = out['Avg Return'] - out['Baseline Avg Return']
    out['Delta Win Rate'] = out['Win Rate'] - out['Baseline Win Rate']
    out['Delta Sharpe'] = out['Sharpe'] - out['Baseline Sharpe']
    out['Delta Sharpe Annualized'] = out['Sharpe_annualized'] - out['Baseline Sharpe Annualized']
    out['Delta MDD'] = out['Max Drawdown'] - out['Baseline Max Drawdown']

    out['Is Active Filter'] = (
        (out['Trade Count'] >= ACTIVE_MIN_TRADES) &
        (out['Delta Trade Count'] < 0) &
        np.isfinite(out['Sharpe']) &
        np.isfinite(out['Baseline Sharpe'])
    )
    out['Is Positive Sharpe'] = out['Delta Sharpe'] > 0
    return out


def evaluate_trade_quality_grid(df, cost_bps=0, c_cols=None, q_values=None):
    c_cols = list(c_cols or c_index_cols.columns)
    q_values = list(q_values or q_list)

    baseline = evaluate_no_filter_with_cost(df, cost_bps=cost_bps)

    rows = []
    for c_col in c_cols:
        for q in q_values:
            m = evaluate_filter_with_cost(df, c_col, quantile=q, cost_bps=cost_bps)
            rows.append(m)

    grid = pd.DataFrame(rows)
    grid = _with_baseline_deltas(grid, pd.Series(baseline))
    return grid


def build_trade_quality_candidate_grid(final_df):
    rows = []
    for cost_bps in cost_bps_list:
        for model_name, sub in final_df.groupby('model'):
            grid = evaluate_trade_quality_grid(sub, cost_bps=cost_bps)
            grid['Model'] = model_name
            grid['Cost (bp)'] = cost_bps
            rows.append(grid)
    return pd.concat(rows, ignore_index=True)


tradeQualityCandidateGrid = build_trade_quality_candidate_grid(finalDf)
activeTradeQualityGrid = tradeQualityCandidateGrid[tradeQualityCandidateGrid['Is Active Filter']].copy()

tradeQualityGridSummary = (
    tradeQualityCandidateGrid
    .groupby(['Model', 'Cost (bp)'])
    .agg(
        Candidate_Count=('C-Index Type', 'count'),
        Active_Candidates=('Is Active Filter', 'sum'),
        Positive_Active_Candidates=('Is Positive Sharpe', lambda x: int((x & tradeQualityCandidateGrid.loc[x.index, 'Is Active Filter']).sum())),
        Mean_Delta_Sharpe=('Delta Sharpe', 'mean'),
        Median_Delta_Sharpe=('Delta Sharpe', 'median'),
        Best_Delta_Sharpe=('Delta Sharpe', 'max'),
        Worst_Delta_Sharpe=('Delta Sharpe', 'min'),
        Mean_Delta_Avg_Return=('Delta Avg Return', 'mean'),
        Best_Delta_Avg_Return=('Delta Avg Return', 'max')
    )
    .reset_index()
)

bestActiveTradeQuality = (
    activeTradeQualityGrid
    .sort_values(['Model', 'Cost (bp)', SELECTION_METRIC], ascending=[True, True, False])
    .groupby(['Model', 'Cost (bp)'])
    .first()
    .reset_index()
)

show_cols = [
    'Model', 'Cost (bp)', 'C-Index Type', 'Quantile',
    'Baseline Trade Count', 'Trade Count', 'Delta Trade Count',
    'Baseline Avg Return', 'Avg Return', 'Delta Avg Return',
    'Baseline Win Rate', 'Win Rate', 'Delta Win Rate',
    'Baseline Sharpe', 'Sharpe', 'Delta Sharpe',
    'Baseline Max Drawdown', 'Max Drawdown', 'Delta MDD',
    'Threshold Value'
]

print('candidate grid:', tradeQualityCandidateGrid.shape)
print('active candidate grid:', activeTradeQualityGrid.shape)
display(tradeQualityGridSummary.round(6))
display(bestActiveTradeQuality[show_cols].round(6))

candidate grid: (216, 28)
active candidate grid: (102, 28)


,Model,Cost (bp),Candidate_Count,Active_Candidates,Positive_Active_Candidates,Mean_Delta_Sharpe,Median_Delta_Sharpe,Best_Delta_Sharpe,Worst_Delta_Sharpe,Mean_Delta_Avg_Return,Best_Delta_Avg_Return
0,gbm,0,24,11,4,-0.013786,0.0,0.135782,-0.155416,-0.043963,0.207688
1,gbm,5,24,11,4,-0.017650,0.0,0.122339,-0.165430,-0.043963,0.207688
2,gbm,10,24,11,4,-0.021514,0.0,0.108896,-0.175443,-0.043963,0.207688
3,mlp,0,24,14,3,-0.016969,0.0,0.083011,-0.095100,-0.034603,0.073933
4,mlp,5,24,14,3,-0.018864,0.0,0.076822,-0.100756,-0.034603,0.073933
5,mlp,10,24,14,3,-0.020759,0.0,0.070633,-0.106411,-0.034603,0.073933
6,rf,0,24,9,8,0.011027,0.0,0.058040,-0.030658,0.025421,0.124192
7,rf,5,24,9,8,0.012097,0.0,0.061719,-0.027279,0.025421,0.124192
8,rf,10,24,9,8,0.013167,0.0,0.065398,-0.023900,0.025421,0.124192


,Model,Cost (bp),C-Index Type,Quantile,Baseline Trade Count,Trade Count,Delta Trade Count,Baseline Avg Return,Avg Return,Delta Avg Return,Baseline Win Rate,Win Rate,Delta Win Rate,Baseline Sharpe,Sharpe,Delta Sharpe,Baseline Max Drawdown,Max Drawdown,Delta MDD,Threshold Value
0,gbm,0,C_RBO_full,0.5,67,32,-35,0.313659,0.373487,0.059828,61.194030,68.750000,7.555970,0.185403,0.321185,0.135782,-6.134243,-2.955633,3.178610,0.463832
1,gbm,5,C_RBO_full,0.5,67,32,-35,0.263659,0.323487,0.059828,61.194030,68.750000,7.555970,0.155848,0.278187,0.122339,-6.434243,-3.005633,3.428610,0.463832
2,gbm,10,C_RBO_full,0.5,67,32,-35,0.213659,0.273487,0.059828,61.194030,68.750000,7.555970,0.126293,0.235189,0.108896,-6.807006,-3.055633,3.751373,0.463832
3,mlp,0,C_tau_ties10,0.4,116,66,-50,0.217741,0.291674,0.073933,59.482759,66.666667,7.183908,0.138153,0.221164,0.083011,-9.376242,-7.961615,1.414627,0.084211
4,mlp,5,C_tau_ties10,0.4,116,66,-50,0.167741,0.241674,0.073933,59.482759,66.666667,7.183908,0.106429,0.183251,0.076822,-9.976242,-8.461615,1.514627,0.084211
5,mlp,10,C_tau_ties10,0.4,116,66,-50,0.117741,0.191674,0.073933,59.482759,66.666667,7.183908,0.074705,0.145338,0.070633,-10.576242,-8.961615,1.614627,0.084211
6,rf,0,C_tau_ties3,0.5,172,97,-75,0.200919,0.325110,0.124192,57.558140,61.855670,4.297531,0.132599,0.190639,0.058040,-13.807412,-10.993207,2.814205,0.102564
7,rf,5,C_tau_ties3,0.5,172,97,-75,0.150919,0.275110,0.124192,57.558140,61.855670,4.297531,0.099601,0.161320,0.061719,-14.607412,-11.293207,3.314205,0.102564
8,rf,10,C_tau_ties3,0.5,172,97,-75,0.100919,0.225110,0.124192,57.558140,61.855670,4.297531,0.066603,0.132001,0.065398,-15.407412,-11.593207,3.814205,0.102564


### Block bootstrap: best active filter의 신뢰구간

각 model-cost pair에서 `Delta Sharpe` 기준으로 선택된 best active candidate를 고정하고, 날짜를 20 trading-day block 단위로 재표본추출한다.

목적은 선택된 후보의 개선폭이 표본 변동에 얼마나 민감한지 보는 것이다. 여기서는 `Delta Sharpe`, `Delta Avg Return`, `Delta Win Rate`, `Delta MDD`의 95% CI를 계산한다.

In [10]:
def sample_dates_by_blocks(unique_dates, rng, block_length=BLOCK_LENGTH_DAYS):
    dates = pd.to_datetime(pd.Series(unique_dates).dropna().sort_values().unique())
    n = len(dates)
    sampled = []
    while len(sampled) < n:
        start = int(rng.integers(0, n))
        for k in range(block_length):
            sampled.append(dates[(start + k) % n])
            if len(sampled) >= n:
                break
    return sampled[:n]


def bootstrap_sample_by_date_blocks(df, rng, block_length=BLOCK_LENGTH_DAYS):
    sampled_dates = sample_dates_by_blocks(df['date'].unique(), rng, block_length=block_length)
    pieces = []
    for boot_idx, d in enumerate(sampled_dates):
        block = df[df['date'] == d].copy()
        block['_bootstrap_order'] = boot_idx
        pieces.append(block)
    if not pieces:
        return df.iloc[0:0].copy()
    out = pd.concat(pieces, ignore_index=True)
    return out.sort_values(['_bootstrap_order', 'asset']).drop(columns=['_bootstrap_order'])


def evaluate_one_policy(df, c_col, q, cost_bps):
    baseline = evaluate_no_filter_with_cost(df, cost_bps=cost_bps)
    candidate = evaluate_filter_with_cost(df, c_col, quantile=q, cost_bps=cost_bps)
    return _with_baseline_deltas(pd.DataFrame([candidate]), pd.Series(baseline)).iloc[0]


def bootstrap_best_active_trade_quality(final_df, best_table, n_bootstrap=N_BOOTSTRAP, block_length=BLOCK_LENGTH_DAYS, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = []

    for _, best in best_table.iterrows():
        model_name = best['Model']
        cost_bps = int(best['Cost (bp)'])
        c_col = best['C-Index Type']
        q = float(best['Quantile'])
        sub = final_df[final_df['model'] == model_name].copy()

        for b in range(n_bootstrap):
            boot = bootstrap_sample_by_date_blocks(sub, rng, block_length=block_length)
            m = evaluate_one_policy(boot, c_col=c_col, q=q, cost_bps=cost_bps)
            rows.append({
                'Model': model_name,
                'Cost (bp)': cost_bps,
                'C-Index Type': c_col,
                'Quantile': q,
                'Bootstrap Iteration': b,
                'Delta Sharpe': m['Delta Sharpe'],
                'Delta Avg Return': m['Delta Avg Return'],
                'Delta Win Rate': m['Delta Win Rate'],
                'Delta MDD': m['Delta MDD'],
                'Trade Count': m['Trade Count'],
                'Delta Trade Count': m['Delta Trade Count']
            })

        print(f'[bootstrap done] model={model_name} cost={cost_bps} c={c_col} q={q}')

    return pd.DataFrame(rows)


def summarize_trade_quality_bootstrap(boot_df):
    rows = []
    metrics = ['Delta Sharpe', 'Delta Avg Return', 'Delta Win Rate', 'Delta MDD']
    for keys, grp in boot_df.groupby(['Model', 'Cost (bp)', 'C-Index Type', 'Quantile']):
        row = dict(zip(['Model', 'Cost (bp)', 'C-Index Type', 'Quantile'], keys))
        for metric in metrics:
            vals = grp[metric].replace([np.inf, -np.inf], np.nan).dropna()
            row[f'{metric} CI Low'] = float(vals.quantile(0.025)) if len(vals) else np.nan
            row[f'{metric} CI High'] = float(vals.quantile(0.975)) if len(vals) else np.nan
            row[f'{metric} Boot Mean'] = float(vals.mean()) if len(vals) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


tradeQualityBootstrapDistribution = bootstrap_best_active_trade_quality(finalDf, bestActiveTradeQuality)
tradeQualityBootstrapSummary = summarize_trade_quality_bootstrap(tradeQualityBootstrapDistribution)

display(tradeQualityBootstrapSummary.round(6))

[bootstrap done] model=gbm cost=0 c=C_RBO_full q=0.5
[bootstrap done] model=gbm cost=5 c=C_RBO_full q=0.5
[bootstrap done] model=gbm cost=10 c=C_RBO_full q=0.5
[bootstrap done] model=mlp cost=0 c=C_tau_ties10 q=0.4
[bootstrap done] model=mlp cost=5 c=C_tau_ties10 q=0.4
[bootstrap done] model=mlp cost=10 c=C_tau_ties10 q=0.4
[bootstrap done] model=rf cost=0 c=C_tau_ties3 q=0.5
[bootstrap done] model=rf cost=5 c=C_tau_ties3 q=0.5
[bootstrap done] model=rf cost=10 c=C_tau_ties3 q=0.5


,Model,Cost (bp),C-Index Type,Quantile,Delta Sharpe CI Low,Delta Sharpe CI High,Delta Sharpe Boot Mean,Delta Avg Return CI Low,Delta Avg Return CI High,Delta Avg Return Boot Mean,Delta Win Rate CI Low,Delta Win Rate CI High,Delta Win Rate Boot Mean,Delta MDD CI Low,Delta MDD CI High,Delta MDD Boot Mean
0,gbm,0,C_RBO_full,0.5,-0.226295,0.494348,0.125614,-0.553683,0.355646,0.011645,-6.000502,20.069898,6.945202,0.000000,14.639248,5.145723
1,gbm,5,C_RBO_full,0.5,-0.201193,0.503063,0.113622,-0.451664,0.366912,0.016420,-5.123517,19.663050,6.928141,0.000000,16.166554,5.771989
2,gbm,10,C_RBO_full,0.5,-0.227978,0.458033,0.103470,-0.485808,0.355567,0.016216,-5.629476,19.677493,7.047493,0.000000,15.772450,6.174078
3,mlp,0,C_tau_ties10,0.4,-0.080482,0.274424,0.085417,-0.167706,0.310434,0.076668,0.257711,14.785101,7.190152,-1.834088,16.582274,5.315613
4,mlp,5,C_tau_ties10,0.4,-0.081635,0.274880,0.084461,-0.164098,0.313773,0.080939,0.356552,14.871854,7.322070,-1.584822,17.504413,5.993721
5,mlp,10,C_tau_ties10,0.4,-0.094150,0.277020,0.078221,-0.173872,0.321214,0.080786,0.189603,15.156889,7.307417,-1.962652,19.509157,6.634483
6,rf,0,C_tau_ties3,0.5,-0.041705,0.207448,0.063191,-0.045005,0.314294,0.120196,-2.775212,11.295317,4.124989,-1.064667,13.008744,4.638477
7,rf,5,C_tau_ties3,0.5,-0.031536,0.214453,0.068504,-0.039136,0.332411,0.125055,-1.895354,12.277816,4.339244,-0.602248,14.266916,5.529216
8,rf,10,C_tau_ties3,0.5,-0.032876,0.209027,0.070080,-0.043063,0.322812,0.120925,-2.193818,11.901592,4.206491,0.000000,16.465906,6.600588


### Block permutation + max-stat correction

귀무가설은 다음과 같다.

```text
C-index 값은 미래 거래 성과와 무관하다.
```

검정 절차는 다음과 같다.

1. 날짜 block 단위로 C-index columns만 permutation한다.
2. `ret_1d_next`, `y_hat`, `date`, `asset` 구조는 유지한다.
3. 각 permutation에서 24개 후보를 모두 평가한다.
4. active filter 조건을 동일하게 적용한다.
5. 그 permutation 안에서 최대 `Delta Sharpe`를 저장한다.
6. 실제 observed best `Delta Sharpe`가 이 null max distribution에서 얼마나 극단적인지 adjusted p-value로 계산한다.

즉 단순히 best filter와 No Filter를 비교하는 것이 아니라, 24개 후보를 뒤져서 best를 고르면 우연히 이 정도 개선이 나올 수 있는가를 검정한다.

In [11]:
def make_date_blocks_for_permutation(df, block_length=BLOCK_LENGTH_DAYS):
    df_sorted = df.sort_values(['date', 'asset']).copy()
    unique_dates = pd.to_datetime(pd.Series(df_sorted['date'].dropna().unique())).sort_values().to_numpy()
    blocks = []
    for i in range(0, len(unique_dates), block_length):
        block_dates = set(unique_dates[i:i + block_length])
        idx = df_sorted.index[df_sorted['date'].isin(block_dates)].to_numpy()
        blocks.append(idx)
    return df_sorted, blocks


def permute_cindex_by_date_blocks(df, c_cols, rng, block_length=BLOCK_LENGTH_DAYS):
    df_perm, blocks = make_date_blocks_for_permutation(df, block_length=block_length)
    if len(blocks) <= 1:
        return df_perm

    order = rng.permutation(len(blocks))
    target_idx = np.concatenate(blocks)
    source_idx = np.concatenate([blocks[i] for i in order])

    out = df_perm.copy()
    out.loc[target_idx, c_cols] = df_perm.loc[source_idx, c_cols].to_numpy()
    return out.sort_index()


def _pvalue_from_null(observed, null_values):
    null_values = pd.Series(null_values).replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
    if len(null_values) == 0 or not np.isfinite(observed):
        return np.nan
    return float((1 + np.sum(null_values >= observed)) / (len(null_values) + 1))


def permutation_max_stat_trade_quality(final_df, best_table, n_permutations=N_PERMUTATIONS, block_length=BLOCK_LENGTH_DAYS, seed=SEED):
    rng = np.random.default_rng(seed)
    p_rows = []
    null_rows = []

    best_lookup = {
        (row['Model'], int(row['Cost (bp)'])): row
        for _, row in best_table.iterrows()
    }

    for model_name, sub in final_df.groupby('model'):
        sub = sub.copy()
        c_cols = list(c_index_cols.columns)

        for cost_bps in cost_bps_list:
            if (model_name, int(cost_bps)) not in best_lookup:
                print(f'[skip] no active observed candidate: model={model_name} cost={cost_bps}')
                continue

            observed_best = best_lookup[(model_name, int(cost_bps))]
            observed_metric = float(observed_best[SELECTION_METRIC])
            selected_null = []
            max_null = []

            selected_c = observed_best['C-Index Type']
            selected_q = float(observed_best['Quantile'])

            for p in range(n_permutations):
                perm = permute_cindex_by_date_blocks(sub, c_cols, rng, block_length=block_length)
                perm_grid = evaluate_trade_quality_grid(perm, cost_bps=cost_bps)
                perm_active = perm_grid[perm_grid['Is Active Filter']].copy()

                if perm_active.empty:
                    selected_val = np.nan
                    max_val = np.nan
                else:
                    max_val = float(perm_active[SELECTION_METRIC].max())
                    selected_match = perm_active[
                        (perm_active['C-Index Type'] == selected_c) &
                        (np.isclose(perm_active['Quantile'].astype(float), selected_q))
                    ]
                    selected_val = float(selected_match[SELECTION_METRIC].iloc[0]) if len(selected_match) else np.nan

                selected_null.append(selected_val)
                max_null.append(max_val)
                null_rows.append({
                    'Model': model_name,
                    'Cost (bp)': cost_bps,
                    'Permutation': p,
                    'Selected Candidate Null': selected_val,
                    'Max Active Null': max_val
                })

            raw_p = _pvalue_from_null(observed_metric, selected_null)
            adjusted_p = _pvalue_from_null(observed_metric, max_null)

            p_rows.append({
                'Model': model_name,
                'Cost (bp)': cost_bps,
                'C-Index Type': selected_c,
                'Quantile': selected_q,
                'Observed Best Metric': observed_metric,
                'Raw p': raw_p,
                'Adjusted p (max-stat)': adjusted_p,
                'Null Selected Mean': float(pd.Series(selected_null).mean()),
                'Null Selected 95%': float(pd.Series(selected_null).quantile(0.95)),
                'Null Max Mean': float(pd.Series(max_null).mean()),
                'Null Max 95%': float(pd.Series(max_null).quantile(0.95)),
                'N Permutations': n_permutations,
                'Block Length Days': block_length
            })

            print(f'[permutation done] model={model_name} cost={cost_bps} raw_p={raw_p:.4f} adj_p={adjusted_p:.4f}')

    return pd.DataFrame(null_rows), pd.DataFrame(p_rows)


tradeQualityPermutationNullTable, tradeQualityPValueTable = permutation_max_stat_trade_quality(finalDf, bestActiveTradeQuality)

display(tradeQualityPValueTable.round(6))

[permutation done] model=gbm cost=0 raw_p=0.1064 adj_p=0.2269
[permutation done] model=gbm cost=5 raw_p=0.1129 adj_p=0.2609
[permutation done] model=gbm cost=10 raw_p=0.1469 adj_p=0.3198
[permutation done] model=mlp cost=0 raw_p=0.1144 adj_p=0.4998
[permutation done] model=mlp cost=5 raw_p=0.1414 adj_p=0.5522
[permutation done] model=mlp cost=10 raw_p=0.1604 adj_p=0.6052
[permutation done] model=rf cost=0 raw_p=0.1289 adj_p=0.4528
[permutation done] model=rf cost=5 raw_p=0.1309 adj_p=0.4308
[permutation done] model=rf cost=10 raw_p=0.1264 adj_p=0.4193


,Model,Cost (bp),C-Index Type,Quantile,Observed Best Metric,Raw p,Adjusted p (max-stat),Null Selected Mean,Null Selected 95%,Null Max Mean,Null Max 95%,N Permutations,Block Length Days
0,gbm,0,C_RBO_full,0.5,0.135782,0.106447,0.226887,0.008595,0.183591,0.083493,0.219640,2000,20
1,gbm,5,C_RBO_full,0.5,0.122339,0.112944,0.260870,0.003708,0.167823,0.080026,0.213014,2000,20
2,gbm,10,C_RBO_full,0.5,0.108896,0.146927,0.319840,-0.002184,0.167834,0.078088,0.213304,2000,20
3,mlp,0,C_tau_ties10,0.4,0.083011,0.114443,0.499750,0.000101,0.115640,0.088784,0.193729,2000,20
4,mlp,5,C_tau_ties10,0.4,0.076822,0.141429,0.552224,0.000510,0.117688,0.089001,0.196716,2000,20
5,mlp,10,C_tau_ties10,0.4,0.070633,0.160420,0.605197,-0.003194,0.121535,0.090827,0.191277,2000,20
6,rf,0,C_tau_ties3,0.5,0.058040,0.128936,0.452774,-0.010005,0.093021,0.055359,0.145909,2000,20
7,rf,5,C_tau_ties3,0.5,0.061719,0.130935,0.430785,-0.008295,0.093492,0.055934,0.148258,2000,20
8,rf,10,C_tau_ties3,0.5,0.065398,0.126437,0.419290,-0.007751,0.094391,0.058094,0.147744,2000,20


### Final trade-quality test table

이 최종표는 기존 거래비용 결과표와 통계검정 결과를 합친다.

해석 기준은 다음과 같다.

- `Adjusted p < 0.05`: 24개 후보 사후 선택 보정 후에도 유의
- `0.05 <= Adjusted p < 0.10`: 보정 후 약한/suggestive evidence
- `Raw p`만 낮고 `Adjusted p`가 높음: best 후보는 좋아 보이나 multiple testing correction 후 비유의
- 둘 다 높음: exploratory finding으로만 해석

In [12]:
def _decision_label(p):
    if pd.isna(p):
        return 'no valid active candidate'
    if p < 0.05:
        return 'significant after max-stat correction'
    if p < 0.10:
        return 'suggestive after max-stat correction'
    return 'not significant after correction'


tradeQualityFinalTestTable = (
    bestActiveTradeQuality[show_cols]
    .merge(
        tradeQualityBootstrapSummary,
        on=['Model', 'Cost (bp)', 'C-Index Type', 'Quantile'],
        how='left'
    )
    .merge(
        tradeQualityPValueTable,
        on=['Model', 'Cost (bp)', 'C-Index Type', 'Quantile'],
        how='left'
    )
)

tradeQualityFinalTestTable['Decision'] = tradeQualityFinalTestTable['Adjusted p (max-stat)'].apply(_decision_label)

final_cols = [
    'Model', 'Cost (bp)', 'C-Index Type', 'Quantile',
    'Baseline Trade Count', 'Trade Count', 'Delta Trade Count',
    'Delta Avg Return', 'Delta Avg Return CI Low', 'Delta Avg Return CI High',
    'Delta Sharpe', 'Delta Sharpe CI Low', 'Delta Sharpe CI High',
    'Delta Win Rate', 'Delta Win Rate CI Low', 'Delta Win Rate CI High',
    'Delta MDD', 'Delta MDD CI Low', 'Delta MDD CI High',
    'Raw p', 'Adjusted p (max-stat)', 'Decision'
]

tradeQualityFinalTestTable = tradeQualityFinalTestTable[final_cols].sort_values(['Cost (bp)', 'Model']).reset_index(drop=True)

display(tradeQualityFinalTestTable.round(6))

,Model,Cost (bp),C-Index Type,Quantile,Baseline Trade Count,Trade Count,Delta Trade Count,Delta Avg Return,Delta Avg Return CI Low,Delta Avg Return CI High,...,Delta Sharpe CI High,Delta Win Rate,Delta Win Rate CI Low,Delta Win Rate CI High,Delta MDD,Delta MDD CI Low,Delta MDD CI High,Raw p,Adjusted p (max-stat),Decision
0,gbm,0,C_RBO_full,0.5,67,32,-35,0.059828,-0.553683,0.355646,...,0.494348,7.555970,-6.000502,20.069898,3.178610,0.000000,14.639248,0.106447,0.226887,not significant after correction
1,mlp,0,C_tau_ties10,0.4,116,66,-50,0.073933,-0.167706,0.310434,...,0.274424,7.183908,0.257711,14.785101,1.414627,-1.834088,16.582274,0.114443,0.499750,not significant after correction
2,rf,0,C_tau_ties3,0.5,172,97,-75,0.124192,-0.045005,0.314294,...,0.207448,4.297531,-2.775212,11.295317,2.814205,-1.064667,13.008744,0.128936,0.452774,not significant after correction
3,gbm,5,C_RBO_full,0.5,67,32,-35,0.059828,-0.451664,0.366912,...,0.503063,7.555970,-5.123517,19.663050,3.428610,0.000000,16.166554,0.112944,0.260870,not significant after correction
4,mlp,5,C_tau_ties10,0.4,116,66,-50,0.073933,-0.164098,0.313773,...,0.274880,7.183908,0.356552,14.871854,1.514627,-1.584822,17.504413,0.141429,0.552224,not significant after correction
5,rf,5,C_tau_ties3,0.5,172,97,-75,0.124192,-0.039136,0.332411,...,0.214453,4.297531,-1.895354,12.277816,3.314205,-0.602248,14.266916,0.130935,0.430785,not significant after correction
6,gbm,10,C_RBO_full,0.5,67,32,-35,0.059828,-0.485808,0.355567,...,0.458033,7.555970,-5.629476,19.677493,3.751373,0.000000,15.772450,0.146927,0.319840,not significant after correction
7,mlp,10,C_tau_ties10,0.4,116,66,-50,0.073933,-0.173872,0.321214,...,0.277020,7.183908,0.189603,15.156889,1.614627,-1.962652,19.509157,0.160420,0.605197,not significant after correction
8,rf,10,C_tau_ties3,0.5,172,97,-75,0.124192,-0.043063,0.322812,...,0.209027,4.297531,-2.193818,11.901592,3.814205,0.000000,16.465906,0.126437,0.419290,not significant after correction


In [13]:
# Save trade-quality statistical test artifacts

ARTIFACT_DIR = CACHE_DIR

objects_to_save = {
    'trade_quality_candidate_grid': tradeQualityCandidateGrid,
    'trade_quality_grid_summary': tradeQualityGridSummary,
    'trade_quality_best_active_table': bestActiveTradeQuality,
    'trade_quality_bootstrap_distribution': tradeQualityBootstrapDistribution,
    'trade_quality_bootstrap_summary': tradeQualityBootstrapSummary,
    'trade_quality_permutation_null_table': tradeQualityPermutationNullTable,
    'trade_quality_pvalue_table': tradeQualityPValueTable,
    'trade_quality_final_test_table': tradeQualityFinalTestTable,
}

for name, obj in objects_to_save.items():
    pkl_path = ARTIFACT_DIR / f'{name}.pkl'
    csv_path = ARTIFACT_DIR / f'{name}.csv'
    obj.to_pickle(pkl_path)
    obj.to_csv(csv_path, index=False)
    print(f'[saved] {pkl_path}')
    print(f'[saved] {csv_path}')

save_df_pretty(tradeQualityGridSummary.round(6), 'trade_quality_grid_summary.png', title='Trade-quality Active Candidate Summary')
save_df_pretty(bestActiveTradeQuality[show_cols].round(6), 'trade_quality_best_active_candidates.png', title='Best Active C-index Filters by Delta Sharpe')
save_df_pretty(tradeQualityFinalTestTable.round(6), 'trade_quality_final_test_table.png', title='Trade-quality Bootstrap and Max-stat Test')

print(f'saved trade-quality statistical test tables to {ARTIFACT_DIR} and {OUTPUT_DIR}')

[saved] /content/drive/MyDrive/c-index/artifacts/trade_quality_candidate_grid.pkl
[saved] /content/drive/MyDrive/c-index/artifacts/trade_quality_candidate_grid.csv
[saved] /content/drive/MyDrive/c-index/artifacts/trade_quality_grid_summary.pkl
[saved] /content/drive/MyDrive/c-index/artifacts/trade_quality_grid_summary.csv
[saved] /content/drive/MyDrive/c-index/artifacts/trade_quality_best_active_table.pkl
[saved] /content/drive/MyDrive/c-index/artifacts/trade_quality_best_active_table.csv
[saved] /content/drive/MyDrive/c-index/artifacts/trade_quality_bootstrap_distribution.pkl
[saved] /content/drive/MyDrive/c-index/artifacts/trade_quality_bootstrap_distribution.csv
[saved] /content/drive/MyDrive/c-index/artifacts/trade_quality_bootstrap_summary.pkl
[saved] /content/drive/MyDrive/c-index/artifacts/trade_quality_bootstrap_summary.csv
[saved] /content/drive/MyDrive/c-index/artifacts/trade_quality_permutation_null_table.pkl
[saved] /content/drive/MyDrive/c-index/artifacts/trade_quality_per

### Interpretation note

이 검정의 결론 문장은 `tradeQualityFinalTestTable`의 `Adjusted p (max-stat)`를 기준으로 작성한다.

- 보정 후 유의하지 않으면: “C-index filter는 거래비용 반영 후 일부 per-trade Sharpe/MDD 개선 경향을 보였으나, 24개 후보 탐색에 따른 max-stat correction 이후 통계적으로 유의한 개선은 확인되지 않았다.”
- 보정 후 유의하면: “active filter 후보와 multiple testing correction을 고려한 뒤에도 일부 model-cost pair에서 C-index filter의 trade-quality improvement가 통계적으로 지지되었다.”

이 섹션은 기존의 row-level paired Delta Mean Return 검정이 아니라, 본 연구의 reliability filter 프레이밍에 맞춘 per-trade quality 검정이다.